# Analysis of uoe5 pipeline run
Reran antibody panel 1 with new updates within the pipeline, resutling in run uoe5.

Major pipeline changes include more quality control mesasures focused on organelle channels (lysosome, endosome, and mitochondria) - I added the contrast check that we run for the nuclues channel to script 3 to check each channel before segmentation is preformed. The organelle channels are generally more dim than the nuclues channel, so the threshold for QC is quite a bit lower than that of the nucleus channel.

The goals here are to preform the prior PCA analyses (plus additional analysis on the genotype level, manual focused analsysis, and structure level analysis) and assess whether we reduced the number of poor quality images that make it to this analysis stage. 

In [24]:
import pathlib as Path
import pandas as pd
import os

import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Metadata validation
Alena aggregated all of the cell culture group's metadata information: iNDI_Plate_ID, Well_ID, Genotype_Name, JAX_ID, Batch_Info, Cell_Line_Info\
We later queried JAX_ID on the Jackson labs webpage (see JAX_iPSC_Catalog.tsv) to aquire all clean annotation which was merged into the metadata sheet - colnames: JAX_Code, Gene, Variant, Genotype, Gene_Var_Type

In [25]:
all_automated_plates_combined_fn = "/data/CARDPB2/iNDI/Production/metadata/all_automated_plates_combined.csv"
all_automated_plates_combined = pd.read_csv(all_automated_plates_combined_fn)

print(all_automated_plates_combined.head())

  iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info  \
0    INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN   
1    INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN   
2    INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN   
3    INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN   
4    INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN   

      JAX_Code     Gene Variant Genotype          Gene_Var_Type  \
0  JIPSC001038  HNRNPA1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
1  JIPSC001038  HNRNPA1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
2  JIPSC001042  HNRNPA1   D262N   REV/WT   HNRNPA1_D262N_REV/WT   
3  JIPSC001038  HNRNPA1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
4  JIPSC001034     PFN1    C71G   SNV/WT       PFN1_C71G_SNV/WT   

                                      Cell_Line_Info  
0  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  
1  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  
2  JAX_ID_J

In [26]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20260901_155524_uoe5/organelle_features/*_organelle_features_*.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())

028ebee9-afaf-4ff8-b435-af11714285dc_organelle_features__20260902_114905.parquet: 116923 rows
07fc5da6-9d7d-4c97-858b-4b76df1859a5_organelle_features__20260902_114905.parquet: 115713 rows
2350e82d-40dd-4bb4-93de-44edec0d1718_organelle_features__20260902_114905.parquet: 138260 rows
2b1c31ad-579d-4ea6-a782-251ea083ed6a_organelle_features__20260902_114905.parquet: 206382 rows
2da14033-ab86-4f33-b4f7-ef375c80ce62_organelle_features__20260902_114905.parquet: 166866 rows
32d75d22-bd17-4629-8257-a4056e3ffcee_organelle_features__20260902_114905.parquet: 107005 rows
4405a3b2-6b88-49b1-91f3-992e09ccbd16_organelle_features__20260902_114905.parquet: 131590 rows
51cf625a-9422-406f-aa9e-a9f1238d1b96_organelle_features__20260902_114905.parquet: 177582 rows
72c0a713-b5d5-4cd4-a826-c556368ed1d1_organelle_features__20260902_114905.parquet: 100645 rows
7e03b502-ad64-4090-a9e2-fc4c3ecdae11_organelle_features__20260902_114905.parquet: 169399 rows
934f6de5-f58b-4a1e-b225-ac6414500bb7_organelle_features__202

In [27]:
## pull in metadata
plate_id_fn = "/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv"
plate_id = pd.read_csv(plate_id_fn)
print(plate_id.head())

automated_plates_fn = "/data/CARDPB2/iNDI/Production/metadata/all_automated_plates_combined.csv"
manual_plates_fn = "/data/CARDPB2/iNDI/Production/metadata/all_manual_plates_combined.csv"

automated_plate_md = pd.read_csv(automated_plates_fn)
manual_plates_md = pd.read_csv(manual_plates_fn)

parts = manual_plates_md["Genotype_Name"].str.split(" ", n=2, expand=True)
is_triple = manual_plates_md["Genotype_Name"].str.split().str.len() == 3
is_parental = manual_plates_md["Genotype_Name"].str.strip() == "Parental"

manual_plates_md["Gene"] = np.select(
    [is_parental, is_triple],
    ["KOLF2.1J", parts[0]],
    default=np.nan,
)
manual_plates_md["Variant"] = np.select(
    [is_parental, is_triple],
    ["NA", parts[1]],
    default=np.nan,
)
manual_plates_md["Common_genotype"] = np.select(
    [is_parental, is_triple],
    ["wild-type", parts[2]],
    default=np.nan,
)

print(automated_plate_md.head())
print(manual_plates_md.head())

auto_cols = automated_plate_md.columns.tolist()
man_cols = manual_plates_md.columns.tolist()

print("Cols only in automated:", [c for c in auto_cols if c not in man_cols])
print("Cols only in manual:   ", [c for c in man_cols if c not in auto_cols])

# label each source before combining
automated_plate_md = automated_plate_md.copy()
manual_plates_md = manual_plates_md.copy()
automated_plate_md["prep_method"] = "automated"
manual_plates_md["prep_method"] = "manual"

# make sure the manual df uses the same column order
manual_plates_md = manual_plates_md.reindex(columns=automated_plate_md.columns)

# append rows
all_plates_md = pd.concat([automated_plate_md, manual_plates_md], ignore_index=True)

print(all_plates_md.columns.tolist())
print(all_plates_md.shape)
print(all_plates_md["prep_method"].value_counts())
print(df.columns.tolist())

print("df Measurement_ID sample:", df["Measurement_ID"].head(3).tolist())
print("plate_id Folder_ID sample:", plate_id["Folder_ID"].head(3).tolist())
print("df dtype:", df["Measurement_ID"].dtype, "| plate_id dtype:", plate_id["Folder_ID"].dtype)

# keep only the mapping columns you need
plate_id_map = plate_id[["Folder_ID", "iNDI_Plate_ID", "Layout"]].drop_duplicates("Folder_ID")

# optional: remove columns in df that would collide
df = df.drop(columns=["Folder_ID", "iNDI_Plate_ID", "Layout"], errors="ignore")

print("Folder_ID unique?", plate_id_map["Folder_ID"].is_unique)
print("Folder_IDs in plate_id:", plate_id_map["Folder_ID"].nunique())

df_ids = set(df["Measurement_ID"].unique())
map_ids = set(plate_id_map["Folder_ID"].unique())
print(f"df experiments: {len(df_ids)} | matched in map: {len(df_ids & map_ids)} | unmatched: {len(df_ids - map_ids)}")

df = df.merge(
    plate_id_map,
    left_on="Measurement_ID",
    right_on="Folder_ID",
    how="left",
    validate="many_to_one",
    suffixes=("", "")  # no overlap now, so no _x/_y
)

print(f"After merge: {len(df)} rows")
print(df.columns.tolist())

# build Well_ID
import string
df["Well_ID"] = df.apply(
    lambda r: f"{string.ascii_uppercase[int(r['Row'])-1]}{int(r['Column']):02d}",
    axis=1,
)

# match-rate on the FULL composite key
df_keys = set(zip(df["iNDI_Plate_ID"], df["Well_ID"]))
md_keys = set(zip(all_plates_md["iNDI_Plate_ID"], all_plates_md["Well_ID"]))
print(f"df keys: {len(df_keys)} | matched: {len(df_keys & md_keys)} | unmatched: {len(df_keys - md_keys)}")

# metadata composite key unique?
print("md plate+well unique?",
      not all_plates_md.duplicated(["iNDI_Plate_ID", "Well_ID"]).any())

df = df.merge(
    all_plates_md,
    on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
    validate="many_to_one",
)
print(f"After merge: {len(df)} rows")   # should be unchanged
print(df.columns.tolist())

  iNDI_Plate_ID    Layout                             Folder_ID
0    INDI00002D  Plate002  2b1c31ad-579d-4ea6-a782-251ea083ed6a
1    INDI00007D  Plate007  d7e04b5c-a253-42ff-859f-2af0e6047a00
2    INDI00011D  Plate011  f8062781-891b-4688-a59e-fccbf48325cf
3    INDI00015D  Plate015  caef83c7-6b44-4b64-b8ec-e8d84ca6759d
4    INDI00019D  Plate019  c8f2dbbe-6632-4063-ab68-0b4fa37909af
  iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info  \
0    INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN   
1    INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN   
2    INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN   
3    INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN   
4    INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN   

      JAX_Code     Gene Variant Genotype          Gene_Var_Type  \
0  JIPSC001038  HNRNPA1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
1  JIPSC001038  HNRNPA1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/S

In [28]:
for plate, grp in df.groupby('iNDI_Plate_ID'):
    print(f"\n{plate}")
    print(grp.groupby('Gene_Var_Type')['Well_ID'].nunique())


INDI00002D
Gene_Var_Type
FUS_R495X_REV/WT     48
FUS_R495X_SNV/SNV    48
FUS_R495X_SNV/WT     48
KOLF2.1J             96
VCP_R159H_REV/WT     48
VCP_R159H_SNV/SNV    48
VCP_R159H_SNV/WT     48
Name: Well_ID, dtype: int64

INDI00007D
Gene_Var_Type
KOLF2.1J                96
TARDBP_M337V_REV/WT     48
TARDBP_M337V_SNV/WT     48
TARDBP_Q331K_SNV/SNV    47
TIA1_P351L_REV/WT       48
TIA1_P351L_SNV/SNV      48
Name: Well_ID, dtype: int64

INDI00011D
Gene_Var_Type
FUS_R216C_SNV/SNV      47
KOLF2.1J               96
PSEN2_N141I_REV/WT     47
PSEN2_N141I_SNV/SNV    48
TBK1_E696K_REV/WT       3
TBK1_E696K_SNV/SNV      4
TBK1_E696K_SNV/WT       2
Name: Well_ID, dtype: int64

INDI00015D
Gene_Var_Type
APP_A673T_REV/WT      48
APP_A673T_SNV/WT      48
APP_E693G_SNV/SNV     48
APP_E693G_SNV/WT      48
APP_V717I_SNV/SNV     48
KOLF2.1J              48
PSEN1_E276A_SNV/WT    48
Name: Well_ID, dtype: int64

INDI00019D
Gene_Var_Type
APP_E693G_REV/WT       41
FUS_R216C_REV/WT       47
FUS_R216C_SNV/WT   

In [29]:
## add new analysis to check number of channels per nucleus is 4 (including nuclear channel) for all nuclei in the dataset - If a nucleus has less than 4 channels remove it from analysis
counts = (
    df.groupby(["DAPI_filename", "Nucleus_ID"])["Structure"]
        .nunique()
)

keep_keys = counts[counts == 3].index


filtered = df.set_index(["DAPI_filename", "Nucleus_ID"]).loc[keep_keys].reset_index()
print(f"Filtered dataset: {len(filtered)} rows (from {len(df)} original rows)")

Filtered dataset: 2959102 rows (from 2963186 original rows)


Reduced by ~ 4500 ROIs

In [30]:
df = filtered  # replace the original df with the filtered one for further analysis

## Apply subjective filters from Marianita

In [31]:
# randomization_batch
randomization_batch_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel1/batches.csv"
randomization_batch = pd.read_csv(randomization_batch_fn)
randomization_batch = randomization_batch[randomization_batch["AbPanel"] == 1].copy()

print("batches columns:", randomization_batch.columns.tolist())
print(randomization_batch.head())
print("\ndf columns:", df.columns.tolist())
print(df.head())

merged = df.merge(
    randomization_batch[["iNDI_Plate_ID", "Batch"]],
    on="iNDI_Plate_ID",
    how="left",
    validate="many_to_one",
)

batches columns: ['iNDI_Plate_ID', 'AbPanel', 'Batch']
  iNDI_Plate_ID  AbPanel  Batch
0    INDI00002D        1      1
2    INDI00007D        1      1
4    INDI00011D        1      1
5    INDI00015D        1      1
7    INDI00019D        1      2

df columns: ['DAPI_filename', 'Nucleus_ID', 'Measurement_ID', 'subdirectory', 'Row', 'Column', 'Frame', 'Plane', 'Time', 'channel_filename', 'Structure', 'Stain', 'frame_std', 'contrast_check', 'area_px', 'organelle_count', 'average_organelle_area', 'roi_occupancy', 'max_f_intensity', 'sum_f_intensity', 'mean_f_intensity', 'median_f_intensity', 'CoefOfVar_intensity', 'area_mean', 'area_median', 'area_std', 'area_p10', 'area_p90', 'perimeter_mean', 'perimeter_median', 'perimeter_std', 'perimeter_p10', 'perimeter_p90', 'eccentricity_mean', 'eccentricity_median', 'eccentricity_std', 'eccentricity_p10', 'eccentricity_p90', 'solidity_mean', 'solidity_median', 'solidity_std', 'solidity_p10', 'solidity_p90', 'extent_mean', 'extent_median', 'extent_s

In [32]:
# MS_filter 
MS_filter_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel1/marianita_excluded_genotypes_by_plate.csv"
MS_filter = pd.read_csv(MS_filter_fn)
print("MS_filter columns:", MS_filter.columns.tolist())
print(MS_filter.head())

def norm_geno(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace("_", " ", regex=False)   # underscores -> spaces
         .str.replace(r"\s+", " ", regex=True)  # collapse repeats
         .str.upper()                            # case-insensitive
    )

remove_pairs = set(zip(
    MS_filter["iNDI_Plate_ID"].astype(str).str.strip(),
    norm_geno(MS_filter["Genotype_Name"]),
))

key = list(zip(
    merged["iNDI_Plate_ID"].astype(str).str.strip(),
    norm_geno(merged["Genotype_Name"]),
))
mask = pd.Series(key, index=merged.index).isin(remove_pairs)

filtered = merged[~mask].copy()
print(f"removed {mask.sum()} rows; {len(filtered)} remain")

# confirm the normalization actually bridged the two formats
missing = remove_pairs - set(key)
if missing:
    print(f"note: {len(missing)} pair(s) still unmatched after normalization:")
    for p in sorted(missing):
        print("  ", p)

MS_filter columns: ['iNDI_Plate_ID', 'Genotype_Name']
  iNDI_Plate_ID    Genotype_Name
0    INDI00007D   TIA1_P351L_HET
1    INDI00008D   TIA1_P351L_HET
2    INDI00011D  PSEN2_N141I_REV
3    INDI00011D   TBK1_E696K_HET
4    INDI00011D   TBK1_E696K_HOM
removed 22895 rows; 2936207 remain
note: 27 pair(s) still unmatched after normalization:
   ('INDI00007D', 'TIA1 P351L HET')
   ('INDI00008D', 'TIA1 P351L HET')
   ('INDI00016D', 'PSEN1 E276A HET')
   ('INDI00020D', 'APP E693G REV')
   ('INDI00020D', 'PSEN1 E276A HOM')
   ('INDI00020D', 'PSEN2 N141I HET')
   ('INDI00023D', 'SNCB P123H REV')
   ('INDI00024D', 'SNCB P123H HET')
   ('INDI00024D', 'SNCB P123H HOM')
   ('INDI00024D', 'SNCB P123H REV')
   ('INDI00026D', 'HNRNPA1 D262N REV')
   ('INDI00026D', 'PFN1 C71G HET')
   ('INDI00028D', 'HNRNPA1 D262N HOM')
   ('INDI00028D', 'HNRNPA1 D262N REV')
   ('INDI00028D', 'PFN1 C71G HET')
   ('INDI00039D', 'SNCB P123H HET')
   ('INDI00040D', 'PSEN2 N141I HET')
   ('INDI00040D', 'SNCB P123H HET')
 

In [33]:
filtered.to_parquet("uoe5_organelle_output.parquet", index=False)

In [46]:
all_automated_plates_combined_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel1/AbPanel1_automated_plates_combined.csv"
all_automated_plates_combined = pd.read_csv(all_automated_plates_combined_fn)
all_automated_plates_combined['JAX_ID'] = (
    all_automated_plates_combined['JAX_ID'].str.replace('JAX_ID_', '', regex=False)
)

JAX_catalog_fn = "/data/CARDPB2/iNDI/Production/metadata/JAX_iPSC_Catalog.tsv"
JAX_catalog = pd.read_csv(JAX_catalog_fn, sep='\t')

# clean up all_automated_plates_combined$JAX_ID column to remove 'JAX_ID_" to merge with JAX_catalog$JAX_Code
merged = all_automated_plates_combined.merge(
    JAX_catalog,
    left_on='JAX_ID',
    right_on='JAX_Code',
    how='left',
    suffixes=('_x', '_y'),
)

# Collapse the overlapping columns back to clean, unsuffixed names.
# Values agree wherever both are present (0 mismatches), so prefer the plate
# side and fall back to the catalog where the plate side is NaN.
for c in ['Gene', 'Variant', 'Genotype', 'Gene_Var_Type']:
    merged[c] = merged[f'{c}_x'].combine_first(merged[f'{c}_y'])
    merged = merged.drop(columns=[f'{c}_x', f'{c}_y'])

# JAX_Code is the merge key duplicated on both sides; keep one, drop the suffixed pair.
merged['JAX_Code'] = merged['JAX_Code_x'].combine_first(merged['JAX_Code_y'])
merged = merged.drop(columns=['JAX_Code_x', 'JAX_Code_y'])

print(merged.columns.tolist())
print(merged.head())

['iNDI_Plate_ID', 'Well_ID', 'Genotype_Name', 'JAX_ID', 'Batch_Info', 'Cell_Line_Info', 'Gene', 'Variant', 'Genotype', 'Gene_Var_Type', 'JAX_Code']
  iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info  \
0    INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN   
1    INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN   
2    INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN   
3    INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN   
4    INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN   

                                      Cell_Line_Info     Gene Variant  \
0  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  HNRNPA1   D262N   
1  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  HNRNPA1   D262N   
2  JAX_ID_JIPSC001042\nHNRNPA1 D262N REV\nLow cou...  HNRNPA1   D262N   
3  JAX_ID_JIPSC001038\nHNRNPA1 D262N HOM\nLow cou...  HNRNPA1   D262N   
4  JAX_ID_JIPSC001034\nPFN1 C71G HET\nLow counts ...     PFN1    C

In [48]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20260901_155524_uoe5/nuclei_filtered/*_nuclei_filtered.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())

plate_id_fn = "/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv"
plate_id = pd.read_csv(plate_id_fn)
print(plate_id.head())

df = df.merge(
    plate_id,
    left_on="Measurement_ID",
    right_on="Folder_ID",
    how="left",
    validate="many_to_one",
)

# Row/Column are 1-based ints from the filename parse (r01c01...).
df["Well_ID"] = (
    df["Row"].astype(int).map(lambda r: string.ascii_uppercase[r - 1])
    + df["Column"].astype(int).map(lambda c: f"{c:02d}")
)

df = df.merge(
    merged[["iNDI_Plate_ID", "Well_ID", "Gene_Var_Type", "Genotype_Name", "Gene", "Variant"]],
    on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
    validate="many_to_one",
)

# sanity check the join
n_unmatched = df["Gene_Var_Type"].isna().sum()
print(f"{n_unmatched} nuclei rows didn't match a genotype")
if n_unmatched:
    print(df.loc[df["Gene_Var_Type"].isna(), ["iNDI_Plate_ID", "Well_ID"]].drop_duplicates().head(20))

print(df.head())

df.to_parquet("uoe5_nucleus_output.parquet", index=False)

028ebee9-afaf-4ff8-b435-af11714285dc_nuclei_filtered.parquet: 87928 rows
07fc5da6-9d7d-4c97-858b-4b76df1859a5_nuclei_filtered.parquet: 81886 rows
2350e82d-40dd-4bb4-93de-44edec0d1718_nuclei_filtered.parquet: 98782 rows
2b1c31ad-579d-4ea6-a782-251ea083ed6a_nuclei_filtered.parquet: 124127 rows
2da14033-ab86-4f33-b4f7-ef375c80ce62_nuclei_filtered.parquet: 109822 rows
32d75d22-bd17-4629-8257-a4056e3ffcee_nuclei_filtered.parquet: 68940 rows
4405a3b2-6b88-49b1-91f3-992e09ccbd16_nuclei_filtered.parquet: 90673 rows
51cf625a-9422-406f-aa9e-a9f1238d1b96_nuclei_filtered.parquet: 125952 rows
72c0a713-b5d5-4cd4-a826-c556368ed1d1_nuclei_filtered.parquet: 70820 rows
7e03b502-ad64-4090-a9e2-fc4c3ecdae11_nuclei_filtered.parquet: 112023 rows
934f6de5-f58b-4a1e-b225-ac6414500bb7_nuclei_filtered.parquet: 110118 rows
9697941f-11d4-43bb-8f50-d424c4e05c29_nuclei_filtered.parquet: 109010 rows
b96883e9-18b4-4f0d-ba6e-0ddd08c5413f_nuclei_filtered.parquet: 102498 rows
c170959e-633a-458c-8542-2aa619c0d7f8_nuclei_

In [4]:
import os
import glob
import string

import numpy as np
import pandas as pd

# ==========================================================================
# Paths  (CHECK THESE — see notes below)
# ==========================================================================
META_DIR = "/data/CARDPB2/iNDI/Production/metadata"
PANEL_DIR = f"{META_DIR}/AbPanel1"

AUTO_FN = f"{PANEL_DIR}/AbPanel1_automated_plates_combined.csv"
MAN_FN  = f"{META_DIR}/all_manual_plates_combined.csv"

JAX_FN  = f"{META_DIR}/JAX_iPSC_Catalog.tsv"
PLATE_ID_FN = f"{META_DIR}/indi_plateID_to_folderID.csv"

NUCLEI_GLOB = ("/data/kelpschdj/iNDI/Production/outputs/"
               "run_20260901_155524_uoe5/nuclei_filtered/*_nuclei_filtered.parquet")
OUT_FN = "uoe5_nucleus_output.parquet"

# Columns carried from plate metadata onto each nucleus row.
MD_MERGE_COLS = ["iNDI_Plate_ID", "Well_ID", "prep_method",
                 "Gene", "Variant", "Genotype", "Genotype_Name", "Gene_Var_Type"]


# ==========================================================================
# 1. Automated plate metadata  (+ JAX catalog enrichment, from block 2)
# ==========================================================================
automated = pd.read_csv(AUTO_FN)
automated["JAX_ID"] = automated["JAX_ID"].str.replace("JAX_ID_", "", regex=False)

jax = pd.read_csv(JAX_FN, sep="\t")

automated = automated.merge(
    jax, left_on="JAX_ID", right_on="JAX_Code",
    how="left", suffixes=("_x", "_y"),
)

# Collapse overlapping columns: prefer the plate side, fall back to catalog.
for c in ["Gene", "Variant", "Genotype", "Gene_Var_Type"]:
    automated[c] = automated[f"{c}_x"].combine_first(automated[f"{c}_y"])
    automated = automated.drop(columns=[f"{c}_x", f"{c}_y"])
automated["JAX_Code"] = automated["JAX_Code_x"].combine_first(automated["JAX_Code_y"])
automated = automated.drop(columns=["JAX_Code_x", "JAX_Code_y"])

automated["prep_method"] = "automated"
print(f"[automated] {len(automated)} wells")


# ==========================================================================
# 2. Manual plate metadata  (Gene/Variant parsed from Genotype_Name, block 1)
# ==========================================================================
manual = pd.read_csv(MAN_FN)

parts = manual["Genotype_Name"].str.split(" ", n=2, expand=True)
is_triple = manual["Genotype_Name"].str.split().str.len() == 3
is_parental = manual["Genotype_Name"].str.strip() == "Parental"

manual["Gene"] = np.select(
    [is_parental, is_triple], ["KOLF2.1J", parts[0]], default=np.nan,
)
manual["Variant"] = np.select(
    [is_parental, is_triple], ["NA", parts[1]], default=np.nan,
)
# manual encodes the descriptive genotype here; no SNV/* style column exists,
# so `Genotype` stays NaN for manual (the manual PCA scheme keys off
# Genotype_Name's trailing HOM/HET/REV token, not Genotype).
manual["Common_genotype"] = np.select(
    [is_parental, is_triple], ["wild-type", parts[2]], default=np.nan,
)
manual["Gene_Var_Type"] = np.nan   # only the automated/JAX side defines this

manual["prep_method"] = "manual"
print(f"[manual]    {len(manual)} wells")


# ==========================================================================
# 3. Combine both preps  (union of columns; nothing dropped)
# ==========================================================================
all_plates_md = pd.concat([automated, manual], ignore_index=True)
print("[combined] prep_method counts:")
print(all_plates_md["prep_method"].value_counts())

# metadata composite key must be unique for the many_to_one join below
dup = all_plates_md.duplicated(["iNDI_Plate_ID", "Well_ID"]).sum()
print(f"[combined] duplicate (plate, well) keys: {dup}")
assert dup == 0, "metadata has duplicate (iNDI_Plate_ID, Well_ID) keys"


# ==========================================================================
# 4. Load nuclei-filtered features
# ==========================================================================
files = glob.glob(NUCLEI_GLOB)
parts_ = []
for f in files:
    part = pd.read_parquet(f)
    parts_.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts_, ignore_index=True)
print(f"\nLoaded {len(files)} files, {len(df)} total rows")


# ==========================================================================
# 5. Folder_ID -> iNDI_Plate_ID
# ==========================================================================
plate_id = pd.read_csv(PLATE_ID_FN)

df = df.drop(columns=["Folder_ID", "iNDI_Plate_ID", "Layout"], errors="ignore")
df = df.merge(
    plate_id, left_on="Measurement_ID", right_on="Folder_ID",
    how="left", validate="many_to_one",
)

df_ids = set(df["Measurement_ID"].unique())
map_ids = set(plate_id["Folder_ID"].unique())
print(f"experiments: {len(df_ids)} | matched: {len(df_ids & map_ids)} | "
      f"unmatched: {len(df_ids - map_ids)}")


# ==========================================================================
# 6. Well_ID  (Row/Column are 1-based ints from the r##c## filename parse)
# ==========================================================================
df["Well_ID"] = (
    df["Row"].astype(int).map(lambda r: string.ascii_uppercase[r - 1])
    + df["Column"].astype(int).map(lambda c: f"{c:02d}")
)


# ==========================================================================
# 7. Join genotype + prep metadata onto each nucleus
# ==========================================================================
# match-rate on the composite key BEFORE merging (diagnostic)
df_keys = set(zip(df["iNDI_Plate_ID"], df["Well_ID"]))
md_keys = set(zip(all_plates_md["iNDI_Plate_ID"], all_plates_md["Well_ID"]))
print(f"nucleus keys: {len(df_keys)} | matched: {len(df_keys & md_keys)} | "
      f"unmatched: {len(df_keys - md_keys)}")

df = df.merge(
    all_plates_md[MD_MERGE_COLS],
    on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
    validate="many_to_one",
)
print(f"After merge: {len(df)} rows")   # should be unchanged


# ==========================================================================
# 8. Sanity checks
# ==========================================================================
n_unmatched = df["prep_method"].isna().sum()
print(f"\n{n_unmatched} nuclei rows didn't match any plate metadata")
if n_unmatched:
    print(df.loc[df["prep_method"].isna(),
                 ["iNDI_Plate_ID", "Well_ID"]].drop_duplicates().head(20))

print("\nprep_method distribution across nuclei:")
print(df["prep_method"].value_counts(dropna=False))


# ==========================================================================
# 9. Write
# ==========================================================================
df.to_parquet(OUT_FN, index=False)
print(f"\nWrote {OUT_FN}  ({len(df)} rows, {df.shape[1]} cols)")

[automated] 4992 wells
[manual]    3072 wells
[combined] prep_method counts:
prep_method
automated    4992
manual       3072
Name: count, dtype: int64
[combined] duplicate (plate, well) keys: 0
028ebee9-afaf-4ff8-b435-af11714285dc_nuclei_filtered.parquet: 87928 rows
07fc5da6-9d7d-4c97-858b-4b76df1859a5_nuclei_filtered.parquet: 81886 rows
2350e82d-40dd-4bb4-93de-44edec0d1718_nuclei_filtered.parquet: 98782 rows
2b1c31ad-579d-4ea6-a782-251ea083ed6a_nuclei_filtered.parquet: 124127 rows
2da14033-ab86-4f33-b4f7-ef375c80ce62_nuclei_filtered.parquet: 109822 rows
32d75d22-bd17-4629-8257-a4056e3ffcee_nuclei_filtered.parquet: 68940 rows
4405a3b2-6b88-49b1-91f3-992e09ccbd16_nuclei_filtered.parquet: 90673 rows
51cf625a-9422-406f-aa9e-a9f1238d1b96_nuclei_filtered.parquet: 125952 rows
72c0a713-b5d5-4cd4-a826-c556368ed1d1_nuclei_filtered.parquet: 70820 rows
7e03b502-ad64-4090-a9e2-fc4c3ecdae11_nuclei_filtered.parquet: 112023 rows
934f6de5-f58b-4a1e-b225-ac6414500bb7_nuclei_filtered.parquet: 110118 row